# Offline Learning on Different Replay Buffer Compositions

In [13]:
import torch
import numpy as np
from omegaconf import OmegaConf
from functools import partial
import gymnasium as gym
import matplotlib.pyplot as plt
import re
from pathlib import Path
from tqdm.notebook import tqdm

import time
import datetime
import os

import bbrl_utils
from bbrl_utils.notebook import setup_tensorboard
from bbrl.stats import WelchTTest
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from bbrl.utils.replay_buffer import ReplayBuffer

import bbrl_gymnasium

from pmind.algorithms import DQN, DDPG, TD3, OfflineTD3, BBRLStyleTD3, BBRLStyleIQL
from pmind.losses import dqn_compute_critic_loss, ddqn_compute_critic_loss
from pmind.training import (
    run_dqn,
    run_ddpg,
    run_td3,
)
from pmind.replay import (
    collect_policy_transitions,
    collect_uniform_transitions,
    mix_transitions,
    load_rb_files,
    test_rb_uniform_proportions,
    test_rb_noise_levels
)

from pmind.visualization import plot_perf_vs_rb_composition_from_dict

from pmind.config.loader import load_config

bbrl_utils.setup()

OUTPUT_DIR = f"../results/test_rb_compositions-{datetime.datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
os.mkdir(OUTPUT_DIR)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
ENV_NAMES = (
    "CartPoleContinuous-v1",
    # "Pendulum-v1",
    # "MountainCarContinuous-v0",
    # "LunarLanderContinuous-v3",
)
AGENT_CONSTRUCTOR = (TD3, BBRLStyleTD3, BBRLStyleIQL)[1]

BUFFER_SIZE = 200_000

START_WITH_BEST = False
FIRST_ONLY = True

TEST_UNIFORM_PROPORTIONS = True
TEST_NOISE_LEVELS = False

PROPORTIONS = [0.5] #[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
ACTION_NOISES = [0., 1.0]
SEEDS = [103] #[0, 1, 2, 3, 4]

# NOTE: set small values for test:
MAX_STEPS = 20_000 #100_000
NB_EVAL_ENVS = 3 #10
EVAL_INTERVAL = 10 #100
SAVE_RB_POLICY_INTERVAL = 2_000
BATCH_SIZE = 64 #64
NN_ARCHITECTURE = [32,16] #[400,300] [32,16]

In [15]:
for ENV_NAME in ENV_NAMES:
    rb_exploit = load_rb_files(f"../models/{ENV_NAME}/", action_noises=ACTION_NOISES)
    torch.save(rb_exploit, f"../models/{ENV_NAME}/rb-exploit.pt")

In [17]:
REWARDS_TO_TAKE = {
    "CartPoleContinuous-v1":[500],
    "Pendulum-v1": [-233],
    "MountainCarContinuous-v0": [61], #[35, 61, 86]
    }

for ENV_NAME in ENV_NAMES:
    print(f"Testing {ENV_NAME}:")
    cfg = load_config("td3")
    cfg = OmegaConf.create(cfg.environments[ENV_NAME])

    best_reward, _ = torch.load(
        f"../models/{ENV_NAME}/best-policy.pt", weights_only=False
    )

    # Load replay buffers from file
    rb_exploit = torch.load(f"../models/{ENV_NAME}/rb-exploit.pt", weights_only=False)
    rb_unif = torch.load(f"../models/{ENV_NAME}/rb-unif.pt", weights_only=False)

    cfg_offline = OmegaConf.create(cfg)

    cfg_offline.algorithm.n_steps = MAX_STEPS
    cfg_offline.algorithm.max_epochs = None

    cfg_offline.algorithm.batch_size = BATCH_SIZE
    cfg_offline.algorithm.architecture.actor_hidden_size = NN_ARCHITECTURE
    cfg_offline.algorithm.architecture.critic_hidden_size = NN_ARCHITECTURE
    

    cfg_offline.algorithm.eval_interval = EVAL_INTERVAL
    cfg_offline.algorithm.nb_evals = NB_EVAL_ENVS  # nb of evaluation envs in parallel
    
    cfg_offline.algorithm.save_rb_policy_interval = SAVE_RB_POLICY_INTERVAL

    # learning starts immediately for offline:
    cfg_offline.algorithm.learning_starts = None

    # there is no exploration in offline learning
    cfg_offline.algorithm.action_noise = None
    # cfg_offline.algorithm.target_policy_noise = None # NOTE: but there is target policy noise
    
    if TEST_UNIFORM_PROPORTIONS:
        print("TESTING UNIFORM PROPORTIONS")
        for reward, rb_by_prop in sorted(rb_exploit.items(), reverse=START_WITH_BEST):
            if reward not in REWARDS_TO_TAKE[ENV_NAME]:
                continue
            print(f"Policy with reward {reward} :")
            performances = test_rb_uniform_proportions(
                rb_unif,
                rb_by_prop[0], # take action noise = 0
                BUFFER_SIZE,
                PROPORTIONS,
                AGENT_CONSTRUCTOR,
                cfg_offline,
                SEEDS,
            )
            torch.save(performances, f"{OUTPUT_DIR}/uniform_proportions-{ENV_NAME}-scoring-{reward:.0f}")
            
            if FIRST_ONLY:
                print("Skipped intermediate policies")
                break
    
    if TEST_NOISE_LEVELS:
        print("TESTING NOISE LEVELS")
        for reward, rb_by_noise in sorted(rb_exploit.items(), reverse=START_WITH_BEST):
            if reward not in REWARDS_TO_TAKE[ENV_NAME]:
                continue
            print(f"Policy with reward {reward} :")
            performances = test_rb_noise_levels(
                rb_by_noise,
                BUFFER_SIZE,
                AGENT_CONSTRUCTOR,
                cfg_offline,
                SEEDS,
            ) 
            
            torch.save(performances, f"{OUTPUT_DIR}/noise_levels-{ENV_NAME}-scoring-{reward:.0f}")
            
            if FIRST_ONLY:
                print("Skipped intermediate policies")
                break


Testing CartPoleContinuous-v1:
TESTING UNIFORM PROPORTIONS
Policy with reward 500 :
Reward at initialization: 15.0
2026-05-01 16:42.52 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2026-05-01 16:42.52 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2026-05-01 16:42.52 [info     ] Action size has been automatically determined. action_size=1


  0%|          | 0/2000 [00:00<?, ?it/s]

20000
2026-05-01 16:42.52 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=1)
2026-05-01 16:42.52 [warning  ] Skip building models since they're already built.
2026-05-01 16:42.52 [info     ] Directory is created at d3rlpy_logs/TD3_20260501164252
2026-05-01 16:42.52 [info     ] Parameters                     params={'observation_shape': [4], 'action_size': 1, 'config': {'type': 'td3', 'params': {'batch_size': 64, 'gamma': 0.98, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'min_max', 'params': {'minimum': -1.0, 'maximum': 1.0}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'actor_learning_rate': 0.001, 'critic_learning_rate': 0.001, 'actor_optim_fa

KeyboardInterrupt: 